### Install new Libraries

In [17]:
# !pip install ddgs trafilatura -q

### Setup

In [18]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS # duck duck go
import trafilatura
from IPython.display import Markdown, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

### Step 1: Define the Tools

In [19]:
def search_web(query: str):
    """Search the web using DuckDuckGo browser. Return 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 Got results")
    return json.dumps(results, indent=2)

In [20]:
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura."""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f" \u274C Failed to fetch or extract text.")
    return f"Could not extract text from {url}. Try a different source."

### Step 2: Describe as LLM tools

In [21]:
tools = []

In [22]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Return 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant websites"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type": "function", "function": search_web_function})

In [23]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the content of a URL using trafilatura.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type": "function", "function": fetch_url_function})

### Step 3: Tool Call Handler

In [24]:
def handle_tool_call(tool_calls):
    tool_call_results = []
    content = ""

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001f527 Calling function {function_name} with argument: {args}") # for future debugging

        # Route to the appropriate function based on the function name
        if function_name == "search_web":
            result = search_web(args["query"])
            content = f"Search results: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        # elif function_name == "insert_function_name_3":
        #     content = insert_function_name_3(args["message"])
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        }
        tool_call_results.append(tool_call_result)

    return tool_call_results

### Step 4: System Prompt

This tells the LLM *who it is* and *how to behave*. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce

In [25]:
RESEARCH_AGENT_PROMPT = """
You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
It CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move.
"""

### Step 5: The Agentic Loop

In [26]:
def run_research_agent(topic: str, max_iterations: int=10) -> str:
    """
    Run the research agent on a topic and return the research brief.

    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent infinite loops
    
    Return:
    The research brief as a string
    """
    print(f"\n\U0001F50D Starting research on: {topic}")
    print("=" * 60)
    # Initialize conversation messages list with system prompt + research task
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and rpoduce a comprehensive brief:\n\n{topic}"}
    ]

    # LOOP
    iteration = 1
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        # 1. Call the LLM and get response
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        message = response.choices[0].message
        messages.append(message)

        # 2. Check if LLM called tools
        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)

        # 3. Otherwise: no tools were called, read message content
        else: 
            content = message.content # No tool calls, so there is just a normal text response from the LLM
            # Check if DONE, then return
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n\u2705 Research complete!")
                return research_brief
            
            # Otherwise: not yet done, append message
            else:
                print(f"  \U0001F4AD Agent is thinking:")
                pprint(content)
                # Loop continues to next iteration

        # 4. If we're entering the final iteration, force a final answer
        if (iteration == max_iterations -1):
            print("  \u26a0 Safety limit reached. Stopping research in next iteration.")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your research brief now. You MUST respond with DONE: followed by your brief."})

    # Fallback return
    return "Research incomplete. Maximum iterations reache without finalizing brief."

### Let's run it!

In [27]:
brief = run_research_agent("AI in Healthcare in 2030")
display(Markdown(brief))


🔍 Starting research on: AI in Healthcare in 2030

--- Iteration 2 ---
 🔧 Calling function search_web with argument: {'query': 'AI in healthcare in 2030'}
 ✅ Got results

--- Iteration 3 ---
 🔧 Calling function fetch_url with argument: {'url': 'https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade'}
 ✅ Got text: 3275 chars
 🔧 Calling function fetch_url with argument: {'url': 'https://www.weforum.org/stories/health-and-healthcare-systems/future-of-artificial-intelligence-healthcare-delivery/'}
 ❌ Failed to fetch or extract text.
 🔧 Calling function fetch_url with argument: {'url': 'https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030'}
 ✅ Got text: 13481 chars

--- Iteration 4 ---
 🔧 Calling function search_web with argument: {'query': 'challenges and ethical considerations of AI in healthcare 2030'}
 ✅ Got results

--- Iteration 5 ---
 🔧 Calling function fetch_url with argument: {'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8826344/'}
 ✅ Got 

Research Brief: AI in Healthcare in 2030

1. Overview and Main Themes:
Artificial Intelligence (AI) is set to deeply transform healthcare by 2030, shifting the paradigm from reactive to proactive and personalized care. Key expected advances include personalized medicine, gene therapy, early disease diagnostics, predictive analytics, AI-powered virtual assistants, and telehealth expansion. The focus is on improving clinical outcomes, reducing chronic diseases, and addressing aging populations through virtual nursing and intelligent monitoring.

2. Key Facts and Market Statistics:
- The global AI in healthcare market is projected to reach $194.79 billion by 2031, growing at a CAGR of ~40%. (MarketsandMarkets)
- The US AI healthcare market will grow from $6 billion in 2024 to $43 billion by 2030, CAGR 38%. (MarketsandMarkets)
- Over 90% of healthcare organizations have adopted AI tools mainly in imaging and diagnostics.
- 65% of US hospitals use AI predictive analytics for at-risk patient identification; some have seen up to 50% reduction in readmissions.
- FDA has approved 692 AI-enabled medical devices by late 2023, majority in radiology.
- AI-powered remote monitoring reduces hospital readmissions by ~45% in chronic disease populations.
- Clinician documentation time expected to reduce by 50% by 2027 through AI automation.
- Approximately 70% patients are willing to use AI health apps; 64% comfortable with AI virtual nursing assistants.
- Health workforce shortages (estimated 11 million globally by 2030) will be partly eased by AI increasing productivity by 15-30% in some settings.

3. Expected AI Trends by 2030:
- Predictive AI to forecast disease risk, enabling early personalized intervention.
- AI clinical assistants to manage workflows, documentation, and patient triage.
- Telehealth platforms powered by AI to broaden healthcare access, especially for underserved areas.
- Precision medicine integrating genomics, environment, and lifestyle data.
- AI accelerating drug discovery and clinical development timelines.

4. Challenges and Ethical Considerations:
- Data privacy, robust security, and informed patient consent.
- Bias in AI training data risking exacerbation of health disparities (e.g., racial, socioeconomic).
- Need for transparency and explainability to maintain trust.
- Ethical principles: autonomy, beneficence, non-maleficence, justice, accountability.
- Inclusion of diverse populations in AI development to ensure equity.
- High AI infrastructure and implementation costs limit smaller providers, risking widening digital divides.
- Continuous monitoring for algorithmic bias and unintended consequences needed.

5. Leading Market Players:
Microsoft, NVIDIA, GE Healthcare, Philips, Oracle, Google, Medtronic, IBM, Cognizant, Amazon Web Services among major technology leaders; specialized startups also emerging.

6. Conclusion:
By 2030, AI will be integral to healthcare, enabling faster, more precise, and personalized medical care with reduced administrative burden. Responsible and equitable AI adoption supported by ethical frameworks and regulatory standards will be essential to maximize benefits and minimize risks, leading to improved health outcomes globally.

Source URLs:
- https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade
- https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030
- https://www.marketsandmarkets.com/PressReleases/artificial-intelligence-healthcare.asp
- https://www.marketsandmarkets.com/Market-Reports/us-ai-in-healthcare-market-249620865.html
- https://www.cdc.gov/pcd/issues/2024/24_0245.htm
- https://www.tempdev.com/blog/2025/05/28/65-key-ai-in-healthcare-statistics/